In [1]:
import pandas as pd

In [2]:
# read in cleaned data
df = pd.read_csv(
    "https://raw.githubusercontent.com/giovani-gutierrez/animal_shelter_outcomes/refs/heads/main/data/clean.csv"
)

In [ ]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit

# data already in chronological order
X = df.drop(columns=["outcome"])
y = df["outcome"] == "NONLIVE"  # nonlive outcomes are the positive class

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
# baseline model (logistic regression)
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_validate

preprocess = make_column_transformer(
    (
        OneHotEncoder(handle_unknown="ignore"),
        [
            "animal_type",
            "intake_type",
            "intake_month",
            "season",
            "intake_day",
        ],
    ),
    remainder="drop",
)

model = make_pipeline(preprocess, LogisticRegression())

metrics = ["precision", "recall", "f1", "average_precision"]
cv_results = cross_validate(model, X_train, y_train, cv=tscv, scoring=metrics)

print(
    f"Mean Precision: {cv_results['test_precision'].mean():.3f} +/- {cv_results['test_precision'].std():.3f}\n",
    f"Mean Recall: {cv_results['test_recall'].mean():.3f} +/- {cv_results['test_recall'].std():.3f}\n",
    f"Mean F1 Score: {cv_results['test_f1'].mean():.3f} +/- {cv_results['test_f1'].std():.3f}\n",
    f"Mean Average Precision: {cv_results['test_average_precision'].mean():.3f} +/- {cv_results['test_average_precision'].std():.3f}",
)

Mean Precision: 0.628 +/- 0.052
 Mean Recall: 0.268 +/- 0.041
 Mean F1 Score: 0.374 +/- 0.041
 Mean Average Precision: 0.502 +/- 0.043
